In [ ]:
test

In [ ]:
WITH wip_source AS (
    SELECT DISTINCT
        CONCAT('WIP001_', CAST(ty.id AS STRING), '_', CAST(sg.id AS STRING), '_', CAST(st.id AS STRING)) AS form_ques_src_id,
        'WIP001' AS form_ques_src_sys_inst_id,
        CONCAT_WS('_', ty.description, sg.description, st.description) AS form_ques_src_name
    FROM silver_wip_statisticalgroup sg
    LEFT JOIN silver_wip_statisticaltype st
        ON sg.id = st.statistical_group_id
    LEFT JOIN silver_wip_statisticalchoice sc
        ON st.id = sc.statistical_type_id
    LEFT JOIN silver_wip_statistic s
        ON sc.id = s.statistical_choice_id
    LEFT JOIN silver_wip_activityheader ah
        ON s.activity_header_id = ah.id
    LEFT JOIN silver_wip_servicetype ty
        ON ah.service_type_id = ty.id
    WHERE ty.id IS NOT NULL
      AND sg.id IS NOT NULL
      AND st.id IS NOT NULL
)

SELECT
    form_ques_src_id,
    COUNT(*) AS cnt
FROM wip_source
GROUP BY form_ques_src_id
HAVING COUNT(*) > 1;

In [ ]:
WITH wip_source AS (
    SELECT DISTINCT
        CONCAT('WIP001_', CAST(ty.id AS STRING), '_', CAST(sg.id AS STRING), '_', CAST(st.id AS STRING)) AS form_ques_src_id,
        'WIP001' AS form_ques_src_sys_inst_id,
        CONCAT_WS('_', ty.description, sg.description, st.description) AS form_ques_src_name
    FROM silver_wip_statisticalgroup sg
    LEFT JOIN silver_wip_statisticaltype st
        ON sg.id = st.statistical_group_id
    LEFT JOIN silver_wip_statisticalchoice sc
        ON st.id = sc.statistical_type_id
    LEFT JOIN silver_wip_statistic s
        ON sc.id = s.statistical_choice_id
    LEFT JOIN silver_wip_activityheader ah
        ON s.activity_header_id = ah.id
    LEFT JOIN silver_wip_servicetype ty
        ON ah.service_type_id = ty.id
    WHERE ty.id IS NOT NULL
      AND sg.id IS NOT NULL
      AND st.id IS NOT NULL
)

SELECT
    form_ques_src_id,
    COUNT(*) AS cnt,
    COLLECT_SET(form_ques_src_name) AS names
FROM wip_source
GROUP BY form_ques_src_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

In [ ]:
SELECT
    id,
    COUNT(*) AS cnt,
    COLLECT_SET(description) AS descriptions
FROM silver_wip_servicetype
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

In [ ]:
WITH wip_servicetype_dedup AS (
    SELECT
        id,
        COALESCE(
            MAX(CASE 
                    WHEN UPPER(TRIM(description)) <> 'NOT IN USE'
                    THEN description
                END),
            MAX(description)
        ) AS description
    FROM silver_wip_servicetype
    GROUP BY id
),
wip_source AS (
    SELECT DISTINCT
        CONCAT('WIP001_', CAST(ty.id AS STRING), '_', CAST(sg.id AS STRING), '_', CAST(st.id AS STRING)) AS form_ques_src_id,
        'WIP001' AS form_ques_src_sys_inst_id,
        CONCAT_WS('_', ty.description, sg.description, st.description) AS form_ques_src_name
    FROM silver_wip_statisticalgroup sg
    LEFT JOIN silver_wip_statisticaltype st
        ON sg.id = st.statistical_group_id
    LEFT JOIN silver_wip_statisticalchoice sc
        ON st.id = sc.statistical_type_id
    LEFT JOIN silver_wip_statistic s
        ON sc.id = s.statistical_choice_id
    LEFT JOIN silver_wip_activityheader ah
        ON s.activity_header_id = ah.id
    LEFT JOIN wip_servicetype_dedup ty
        ON ah.service_type_id = ty.id
    WHERE ty.id IS NOT NULL
      AND sg.id IS NOT NULL
      AND st.id IS NOT NULL
)

SELECT
    form_ques_src_id,
    COUNT(*) AS cnt
FROM wip_source
GROUP BY form_ques_src_id
HAVING COUNT(*) > 1;

In [ ]:
SELECT
    id,
    COUNT(*) AS cnt,
    COLLECT_SET(description) AS descriptions
FROM silver_wip_statisticalgroup
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

In [ ]:
SELECT
    id,
    statistical_group_id,
    COUNT(*) AS cnt,
    COLLECT_SET(description) AS descriptions
FROM silver_wip_statisticaltype
GROUP BY
    id,
    statistical_group_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

In [ ]:
WITH wip_servicetype_dedup AS (
    SELECT
        id,
        COALESCE(
            MAX(CASE 
                    WHEN UPPER(TRIM(description)) NOT LIKE '%NOT IN USE%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DNU%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DO NOT USE%'
                    THEN description
                END),
            MAX(description)
        ) AS description
    FROM silver_wip_servicetype
    GROUP BY id
),

wip_statisticalgroup_dedup AS (
    SELECT
        id,
        COALESCE(
            MAX(CASE 
                    WHEN UPPER(TRIM(description)) NOT LIKE '%NOT IN USE%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DNU%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DO NOT USE%'
                    THEN description
                END),
            MAX(description)
        ) AS description
    FROM silver_wip_statisticalgroup
    GROUP BY id
),

wip_source AS (
    SELECT DISTINCT
        CONCAT('WIP001_', CAST(ty.id AS STRING), '_', CAST(sg.id AS STRING), '_', CAST(st.id AS STRING)) AS form_ques_src_id,
        'WIP001' AS form_ques_src_sys_inst_id,
        CONCAT_WS('_', ty.description, sg.description, st.description) AS form_ques_src_name
    FROM wip_statisticalgroup_dedup sg
    LEFT JOIN silver_wip_statisticaltype st
        ON sg.id = st.statistical_group_id
    LEFT JOIN silver_wip_statisticalchoice sc
        ON st.id = sc.statistical_type_id
    LEFT JOIN silver_wip_statistic s
        ON sc.id = s.statistical_choice_id
    LEFT JOIN silver_wip_activityheader ah
        ON s.activity_header_id = ah.id
    LEFT JOIN wip_servicetype_dedup ty
        ON ah.service_type_id = ty.id
    WHERE ty.id IS NOT NULL
      AND sg.id IS NOT NULL
      AND st.id IS NOT NULL
)

SELECT
    form_ques_src_id,
    COUNT(*) AS cnt
FROM wip_source
GROUP BY form_ques_src_id
HAVING COUNT(*) > 1;

In [ ]:
WITH wip_servicetype_dedup AS (
    SELECT
        id,
        COALESCE(
            MAX(CASE 
                    WHEN UPPER(TRIM(description)) NOT LIKE '%NOT IN USE%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DNU%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DO NOT USE%'
                    THEN description
                END),
            MAX(description)
        ) AS description
    FROM silver_wip_servicetype
    GROUP BY id
),

wip_statisticalgroup_dedup AS (
    SELECT
        id,
        COALESCE(
            MAX(CASE 
                    WHEN UPPER(TRIM(description)) NOT LIKE '%NOT IN USE%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DNU%'
                     AND UPPER(TRIM(description)) NOT LIKE '%DO NOT USE%'
                    THEN description
                END),
            MAX(description)
        ) AS description
    FROM silver_wip_statisticalgroup
    GROUP BY id
),

wip_source AS (
    SELECT DISTINCT
        CONCAT('WIP001_', CAST(ty.id AS STRING), '_', CAST(sg.id AS STRING), '_', CAST(st.id AS STRING)) AS form_ques_src_id,
        'WIP001' AS form_ques_src_sys_inst_id,
        CONCAT_WS('_', ty.description, sg.description, st.description) AS form_ques_src_name
    FROM wip_statisticalgroup_dedup sg
    LEFT JOIN silver_wip_statisticaltype st
        ON sg.id = st.statistical_group_id
    LEFT JOIN silver_wip_statisticalchoice sc
        ON st.id = sc.statistical_type_id
    LEFT JOIN silver_wip_statistic s
        ON sc.id = s.statistical_choice_id
    LEFT JOIN silver_wip_activityheader ah
        ON s.activity_header_id = ah.id
    LEFT JOIN wip_servicetype_dedup ty
        ON ah.service_type_id = ty.id
    WHERE ty.id IS NOT NULL
      AND sg.id IS NOT NULL
      AND st.id IS NOT NULL
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT form_ques_src_id) AS distinct_src_ids
FROM wip_source;